In [4]:
import streamlit as st
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import percentileofscore
import joblib  # or import your model however it's saved

# Load Data
df_stats = pd.read_excel('ANTHRO.xlsx', sheet_name="Sheet1")
df_position = pd.read_excel('ANTHRO.xlsx', sheet_name="Control_Panel")
df_stats = df_stats[["Name", "ID", "Height", "Hand Size", "Wingspan"]].dropna(how='all')
df_position = df_position.dropna(subset=["Position", "Group"])
df = df_position.merge(df_stats, on=["ID", "Name"])

# Load your trained model
model = joblib.load("bayesian_ridge_model.pkl")  # Update with actual model path

# UCLA colors
UCLA_BLUE = "#2774AE"
UCLA_GOLD = "#FFD100"

# App layout
st.set_page_config(page_title="Recruit Weight Prediction", layout="wide")
st.title("📊 Predict Recruit Weight")

# Session state to manage page transitions
if 'page' not in st.session_state:
    st.session_state.page = "input"

if st.session_state.page == "input":
    st.subheader("Enter Player Metrics")

    col1, col2 = st.columns(2)
    with col1:
        height = st.number_input("Height (e.g., 6003 for 6'0\"3)", min_value=5000.0, max_value=7000.0, step=1.0)
        wingspan = st.number_input("Wingspan (inches)", min_value=60.0, max_value=90.0, step=0.1)
    with col2:
        hand_size = st.number_input("Hand Size (inches)", min_value=5.0, max_value=13.0, step=0.1)
        position = st.selectbox("Position", ['All'] + sorted(df["Position"].dropna().unique()))

    if st.button("Submit"):
        st.session_state.inputs = {
            "Height": height,
            "Wingspan": wingspan,
            "Hand Size": hand_size,
            "Position": position
        }
        st.session_state.page = "results"
        st.experimental_rerun()

elif st.session_state.page == "results":
    inputs = st.session_state.inputs
    df_filtered = df.copy()
    if inputs["Position"] != "All":
        df_filtered = df_filtered[df_filtered["Position"] == inputs["Position"]]

    tabs = st.tabs(["Predicted Weight", "Percentiles", "Model Info"])

    # --- TAB 1: Prediction ---
    with tabs[0]:
        st.header("🏋️ Predicted Weight")
        features = pd.DataFrame([{
            "Height": inputs["Height"],
            "Wingspan": inputs["Wingspan"],
            "Hand Size": inputs["Hand Size"]
        }])
        pred = model.predict(features)[0]
        st.metric("Predicted College Weight (lbs)", f"{pred:.1f}")

        # Optional: Prediction Interval
        rmse = 12.5  # example value
        st.write(f"Estimated interval: **{pred - rmse:.1f} to {pred + rmse:.1f} lbs**")

    # --- TAB 2: Percentiles ---
    with tabs[1]:
        st.header("📈 Percentile Visualizations")
        def plot_percentile(data, value, metric):
            percentile = percentileofscore(data, value)
            fig, ax = plt.subplots()
            sns.kdeplot(data, fill=True, color=UCLA_GOLD, ax=ax)
            ax.axvline(value, color=UCLA_BLUE, linestyle="--")
            ax.text(value + 0.2, ax.get_ylim()[1] * 0.05, f"{value} \n{percentile:.1f}th pct", color=UCLA_BLUE, fontweight='bold')
            ax.set_title(f"{metric} Distribution")
            ax.set_xlabel(metric)
            ax.set_ylabel("Density")
            return fig

        for metric in ["Height", "Hand Size", "Wingspan"]:
            if metric in df_filtered.columns:
                st.pyplot(plot_percentile(df_filtered[metric].dropna(), inputs[metric], metric))

    # --- TAB 3: Model Info ---
    with tabs[2]:
        st.header("🧠 Model Information")
        st.write("This model is a Ridge Regression trained on historical recruit data.")
        st.subheader("Model Metrics")
        st.write(f"**RMSE**: 12.5 lbs")  # Example
        st.write(f"**R² Score**: 0.84")
        st.subheader("Coefficients")
        for feature, coef in zip(model.feature_names_in_, model.coef_):
            st.write(f"- **{feature}**: {coef:.2f}")

    if st.button("🔁 Go Back"):
        st.session_state.page = "input"
        st.experimental_rerun()


2025-07-18 10:53:16.924 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-18 10:53:16.925 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-18 10:53:17.009 
  command:

    streamlit run /Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/ipykernel_launcher.py [ARGUMENTS]
2025-07-18 10:53:17.011 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-18 10:53:17.012 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-18 10:53:17.013 Session state does not function when running a script without `streamlit run`
2025-07-18 10:53:17.014 Thread 'MainThread': missing ScriptRunContext! This warning can be igno

SyntaxError: invalid syntax (1370279190.py, line 1)